## 3. XGBoost Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_arrays, load_participant_ids
from src import metrics
from src import train
from src.models.classical_baseline import XGBoostPredictor, HORIZONS

MODEL_NAME = 'XGBoost'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'xgboost'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'xgboost_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'xgboost_val_stability_summary.json'

In [3]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('train set shape: ', X_train.shape, '\nvaluation set shape: ', X_val.shape, '\ntest set shape: ', X_test.shape)
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

train set shape:  (83532, 24) 
valuation set shape:  (16739, 24) 
test set shape:  (16842, 24)
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"\nValidation results — {MODEL_NAME} | seed={seed}\n")

    model = XGBoostPredictor(random_state=seed).fit(X_train, y_train)

    val_pred, test_pred = model.predict(X_val), model.predict(X_test)
    val_results = metrics.evaluate(val_pred, y_val, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(test_pred, y_test, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
        val_pred, y_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        test_pred, y_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'xgboost_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")


Validation results — XGBoost | seed=7

 15 min | RMSE 11.33 | MAE 7.55 | gRMSE 12.46 | Zone A 96.36% | A+B 99.53%
 30 min | RMSE 18.01 | MAE 11.90 | gRMSE 20.50 | Zone A 89.42% | A+B 99.24%
 60 min | RMSE 25.58 | MAE 17.00 | gRMSE 30.51 | Zone A 80.89% | A+B 98.66%
 90 min | RMSE 29.83 | MAE 20.19 | gRMSE 36.14 | Zone A 75.30% | A+B 98.18%
120 min | RMSE 32.35 | MAE 22.29 | gRMSE 39.49 | Zone A 71.79% | A+B 97.88%

--------------------------------------------------------------------------------

Validation results — XGBoost | seed=14

 15 min | RMSE 11.35 | MAE 7.57 | gRMSE 12.48 | Zone A 96.28% | A+B 99.58%
 30 min | RMSE 17.99 | MAE 11.89 | gRMSE 20.46 | Zone A 89.50% | A+B 99.24%
 60 min | RMSE 25.59 | MAE 17.01 | gRMSE 30.52 | Zone A 80.85% | A+B 98.66%
 90 min | RMSE 29.86 | MAE 20.21 | gRMSE 36.18 | Zone A 75.18% | A+B 98.17%
120 min | RMSE 32.37 | MAE 22.31 | gRMSE 39.52 | Zone A 71.49% | A+B 97.88%

------------------------------------------------------------------------------

In [9]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [10]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [11]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== XGBoost validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.36 +/- 0.03 (2.60) mg/dL   MAE = 7.58 +/- 0.02 (1.73)   gRMSE = 12.50 +/- 0.03 (3.66)   Zone A = 96.29 +/- 0.05 (3.82)%   Zone A+B = 99.55 +/- 0.01 (0.92)%
 30-min:  RMSE = 18.01 +/- 0.01 (4.14) mg/dL   MAE = 11.90 +/- 0.01 (2.63)   gRMSE = 20.50 +/- 0.02 (6.33)   Zone A = 89.42 +/- 0.05 (5.72)%   Zone A+B = 99.23 +/- 0.01 (1.12)%
 60-min:  RMSE = 25.58 +/- 0.02 (7.81) mg/dL   MAE = 17.01 +/- 0.01 (5.03)   gRMSE = 30.48 +/- 0.03 (11.89)   Zone A = 80.85 +/- 0.07 (5.71)%   Zone A+B = 98.68 +/- 0.01 (1.48)%
 90-min:  RMSE = 29.82 +/- 0.03 (10.49) mg/dL   MAE = 20.19 +/- 0.02 (6.90)   gRMSE = 36.14 +/- 0.04 (15.71)   Zone A = 75.27 +/- 0.09 (7.41)%   Zone A+B = 98.17 +/- 0.02 (2.02)%
120-min:  RMSE = 32.35 +/- 0.03 (12.08) mg/dL   MAE = 22.29 +/- 0.02 (8.05)   gRMSE = 39.50 +/- 0.04 (18.19)   Zone A = 71.68 +/- 0.09 (8.33)%   Zone A+B = 97.88 +/- 0.02 (2.44)%


In [12]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/xgboost/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/xgboost/xgboost_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/xgboost/xgboost_stability_summary.json
